In [ ]:
RUTA = "F1/data/raw/ai4i2020.csv"
SEP = None
ENCODING = "utf-8"
INFORME = "F1/docs/informe_dataset.md"

In [ ]:
from pathlib import Path
import pandas as pd

MIN_FILAS, MIN_COLUMNAS = 2000, 12
MIN_ROLES = 3
NULOS_MIN, NULOS_MAX = 1.0, 60.0
MAX_MB = 100

OBLIGATORIOS = {"filas", "columnas", "multivariado", "numerica", "categorica"}

In [ ]:
def clasificar(s: pd.Series) -> str:
    """Devuelve el rol analítico de una variable."""
    s = s.dropna()

    if s.empty:
        return "vacía"

    unicos = s.nunique()
    proporcion_unicos = unicos / len(s)
    es_numerica = pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)

    if pd.api.types.is_datetime64_any_dtype(s):
        return "fecha"

    if unicos == 2:
        return "binaria"

    if es_numerica:
        es_entera = s.mod(1).eq(0).all()

        if es_entera and proporcion_unicos > 0.95 and len(s) > 50:
            return "identificador"

        if es_entera and (unicos <= 20 or proporcion_unicos < 0.05):
            return "discreta"

        return "continua"

    # Variables de texto
    if pd.to_datetime(
        s.head(200),
        errors="coerce",
        format="mixed"
    ).notna().mean() > 0.9:
        return "fecha"

    if s.astype(str).str.len().mean() > 60:
        return "texto libre"

    if proporcion_unicos > 0.95 and len(s) > 50:
        return "identificador"

    return "nominal" if unicos <= 15 else "alta cardinalidad"


# Pruebas del clasificador
assert clasificar(pd.Series([0, 1, 1, 0])) == "binaria"
assert clasificar(pd.Series([1.5, 2.7, 3.1] * 40)) == "continua"
assert clasificar(pd.Series(["Norte", "Sur", "Centro"] * 40)) == "nominal"

print("[OK] Casos de prueba superados")

In [ ]:
ruta = Path(RUTA)
df = None

if not ruta.exists():
    print(f"[PENDIENTE] No se encontró el archivo: {ruta}")
    print("Corrija RUTA en la celda de configuración.")
else:
    try:
        if ruta.suffix.lower() in {".xlsx", ".xls"}:
            df = pd.read_excel(ruta)
        else:
            df = pd.read_csv(
                ruta,
                sep=SEP,
                encoding=ENCODING,
                engine="python" if SEP is None else "c"
            )

        print(f"[OK] {df.shape[0]} filas x {df.shape[1]} columnas")

    except Exception as e:
        print(f"[ERROR] {type(e).__name__}: {e}")
        print('Pruebe con SEP = ";" o ENCODING = "latin-1".')

In [ ]:
import os

print(os.getcwd())


In [ ]:
if df is not None:
    perfil = pd.DataFrame({
        "variable": df.columns,
        "dtype": [str(t) for t in df.dtypes],
        "rol": [clasificar(df[c]) for c in df.columns],
        "unicos": [df[c].nunique() for c in df.columns],
        "pct_nulos": (df.isna().mean() * 100).round(1).values,
    })

    display(perfil)
    print(perfil["rol"].value_counts().to_string())
else:
    perfil = None


In [ ]:
if perfil is not None:
    roles = perfil["rol"].value_counts()
    n = lambda rol: int(roles.get(rol, 0))

    roles_utiles = [
        r for r in roles.index
        if r not in ("identificador", "vacía")
    ]

    numericas = n("continua") + n("discreta")
    categoricas = n("nominal") + n("binaria") + n("ordinal")
    alta = n("alta cardinalidad") + n("texto libre")

    filas, columnas = df.shape
    max_nulos = perfil["pct_nulos"].max()
    peso_mb = ruta.stat().st_size / 1024**2
    duplicadas = int(df.duplicated().sum())

    requisitos = [
        ("filas", f"Al menos {MIN_FILAS} filas",
         filas >= MIN_FILAS, f"{filas} filas"),

        ("columnas", f"Al menos {MIN_COLUMNAS} columnas",
         columnas >= MIN_COLUMNAS, f"{columnas} columnas"),

        ("multivariado", f"Combina al menos {MIN_ROLES} roles analíticos",
         len(roles_utiles) >= MIN_ROLES, ", ".join(roles_utiles)),

        ("numerica", "Al menos una variable numérica",
         numericas >= 1, f"{numericas} numéricas (continuas o discretas)"),

        ("categorica", "Al menos una variable categórica",
         categoricas >= 1, f"{categoricas} categóricas (nominales, binarias u ordinales)"),

        ("faltantes", "Presencia de valores faltantes",
         max_nulos >= NULOS_MIN, f"máximo {max_nulos}% en una variable"),

        ("nulos_extremos", f"Ninguna variable sobre {NULOS_MAX:.0f}% de faltantes",
         max_nulos <= NULOS_MAX, f"máximo {max_nulos}%"),

        ("fecha", "Incluye alguna variable de fecha",
         n("fecha") >= 1, f"{n('fecha')} detectadas"),

        ("alta_cardinalidad", "Incluye texto o categórica de alta cardinalidad",
         alta >= 1, f"{alta} detectadas"),

        ("tamano", f"Archivo bajo {MAX_MB} MB",
         peso_mb <= MAX_MB, f"{peso_mb:.1f} MB"),

        ("duplicados", "Sin filas duplicadas exactas",
         duplicadas == 0,
         "ninguna" if duplicadas == 0 else f"{duplicadas} duplicadas"),
    ]

    chequeo = pd.DataFrame([
        {
            "estado": "OK" if ok else (
                "FALTA" if clave in OBLIGATORIOS else "AVISO"
            ),
            "requisito": req,
            "detalle": det
        }
        for clave, req, ok, det in requisitos
    ])

    display(chequeo)

    apto = "FALTA" not in chequeo["estado"].values
    avisos = int((chequeo["estado"] == "AVISO").sum())

    print(
        "RESULTADO:",
        "el conjunto CUMPLE los requisitos mínimos."
        if apto
        else "el conjunto NO cumple. Revise lo marcado como FALTA."
    )

    if apto and avisos:
        print(
            f"Hay {avisos} aviso(s): no impiden usar el conjunto, "
            "pero conviene leerlos."
        )

else:
    chequeo, apto = None, False

In [ ]:
def a_markdown(tabla: pd.DataFrame) -> str:
    """Convierte un DataFrame en una tabla Markdown."""
    encabezado = "| " + " | ".join(tabla.columns) + " |"
    separador = "| " + " | ".join("---" for _ in tabla.columns) + " |"
    filas = [
        "| " + " | ".join(str(v) for v in fila) + " |"
        for fila in tabla.values
    ]
    return "\n".join([encabezado, separador, *filas])


if perfil is not None and INFORME:
    texto = "\n\n".join([
        "# Validación del dataset — MCDI500",
        f"**Archivo:** `{ruta.name}` · {df.shape[0]} filas × {df.shape[1]} columnas",
        "## Perfil de variables",
        a_markdown(perfil),
        "## Requisitos del curso",
        a_markdown(chequeo),
        "## Resultado",
        "El conjunto **cumple** los requisitos mínimos."
        if apto
        else "El conjunto **no cumple todavía** los requisitos mínimos.",
    ])

    Path(INFORME).write_text(texto, encoding="utf-8")

    print(
        f"Informe escrito en: {INFORME} — "
        "adjúntelo al entregable de la Fase 1."
    )

In [ ]:
CLASIFICACION_EQUIPO = {
    "﻿UDI": "identificador",
    "Product ID": "identificador",
    "Type": "nominal",
    "Air temperature [K]": "continua",
    "Process temperature [K]": "continua",
    "Rotational speed [rpm]": "continua",
    "Torque [Nm]": "continua",
    "Tool wear [min]": "discreta",
    "Machine failure": "binaria",
    "TWF": "binaria",
    "HDF": "binaria",
    "PWF": "binaria",
    "OSF": "binaria",
    "RNF": "binaria",
}

if perfil is not None and CLASIFICACION_EQUIPO:
    comparacion = perfil[["variable", "rol"]].rename(
        columns={"rol": "segun_validador"}
    )

    comparacion["segun_equipo"] = comparacion["variable"].map(
        CLASIFICACION_EQUIPO
    )

    comparacion = comparacion.dropna(subset=["segun_equipo"])

    comparacion["coincide"] = (
        comparacion["segun_equipo"] == comparacion["segun_validador"]
    )

    display(comparacion)

    discrepancias = comparacion.loc[
        ~comparacion["coincide"], "variable"
    ].tolist()

    print(f"Discrepancias: {len(discrepancias)}")

    for v in discrepancias:
        print("  · revisar y justificar:", v)

else:
    print("Complete CLASIFICACION_EQUIPO con el rol que ustedes asignaron.")
    